In [ ]:
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import load_npz, csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree, connected_components
from scipy.special import logsumexp
from scipy.stats import spearmanr
from tqdm.auto import tqdm

# --- repository paths -------------------------------------------------------
# `pip install -e .` from the repository root makes this import work anywhere;
# the fallback covers a plain `jupyter lab` started without installing.
try:
    from scisoc.config import paths
except ModuleNotFoundError:
    _here = Path.cwd().resolve()
    _root = next(p for p in (_here, *_here.parents) if (p / "src" / "scisoc").is_dir())
    sys.path.insert(0, str(_root / "src"))
    from scisoc.config import paths

from scisoc.io import load_node_info

paths.ensure()
print(paths.describe())

# The raw subject tables live outside the repository (external drive).
# Point at them with SCISOC_RAW in the environment or in <repo>/.env, e.g.
#     SCISOC_RAW=/Volumes/My Passport_2/Science-Society/data/processed
NEWS_PATH    = paths.raw_subjects("news")     # news_subject_by_year.pkl
PAPER_PATH   = paths.raw_subjects("paper")    # paper_subject_by_year.pkl

NETWORKS_DIR = paths.networks                 # networks/<source>/adj_<year>.npz + node_info.pkl
BACKBONE_DIR = paths.backbone                 # backbone/<source>_<year>_backbone.parquet
EMB_DIR      = paths.embeddings               # <source>_E.npz from src/dysat/train.py
CP_DIR       = paths.cp_results
PLOT_DIR     = paths.figures

SOURCES = ["news", "paper"]
ALPHAS = [0.01, 0.05, 0.1, 0.2]


### Backbone

In [ ]:
def disparity_alpha(W):
    W = W.tocsr()
    s = np.asarray(W.sum(axis=1)).ravel()
    k = np.diff(W.indptr)

    coo = W.tocoo()
    keep = coo.row < coo.col
    i, j, w = coo.row[keep], coo.col[keep], coo.data[keep]

    def alpha(node, w_):
        kk = k[node]
        p = np.divide(w_, s[node], out=np.zeros_like(w_), where=s[node] > 0)
        a = np.power(1.0 - p, np.maximum(kk - 1, 0))
        return np.where(kk > 1, a, 1.0)

    a_i, a_j = alpha(i, w), alpha(j, w)
    return pd.DataFrame({"i": i, "j": j, "w": w,
                         "alpha_min": np.minimum(a_i, a_j)})

def rca_score(W):
    W = W.tocsr()
    s = np.asarray(W.sum(axis=1)).ravel()
    total = s.sum() / 2.0

    coo = W.tocoo()
    keep = coo.row < coo.col
    i, j, w = coo.row[keep], coo.col[keep], coo.data[keep]

    denom = s[i] * s[j]
    rca = np.divide(w * total, denom, out=np.zeros_like(w), where=denom > 0)
    return pd.DataFrame({"i": i, "j": j, "w": w, "rca": rca})

def extract_backbone(W, density_k=3.0, method="disparity", scored=None):
    """`scored` accepts a precomputed score table so it is not recomputed."""
    if method == "disparity":
        df = disparity_alpha(W) if scored is None else scored
        sort_cols, asc = ["alpha_min", "w"], [True, False]
    elif method == "rca":
        df = rca_score(W) if scored is None else scored
        sort_cols, asc = ["rca", "w"], [False, False]
    else:
        raise ValueError(method)

    if df.empty:
        return df.assign(mst_only=pd.Series(dtype=int))

    n_active = int((np.asarray(W.sum(axis=1)).ravel() > 0).sum())
    n_keep = min(int(round(density_k * n_active)), len(df))

    df = df.sort_values(sort_cols, ascending=asc, kind="mergesort").reset_index(drop=True)
    cut = df.iloc[:n_keep].copy()
    cut["mst_only"] = 0

    neg = csr_matrix((-W.data, W.indices, W.indptr), shape=W.shape)
    mst = minimum_spanning_tree(neg).tocoo()
    mst_pairs = {(min(a, b), max(a, b)) for a, b in zip(mst.row, mst.col)}
    have = set(zip(cut.i.to_numpy(), cut.j.to_numpy()))
    add = mst_pairs - have

    if add:
        rest = df.iloc[n_keep:]
        idx = pd.MultiIndex.from_arrays([rest.i, rest.j])
        extra = rest[idx.isin(list(add))].copy()
        extra["mst_only"] = 1
        cut = pd.concat([cut, extra], ignore_index=True)
    return cut

In [ ]:
info = {s: load_node_info(s, NETWORKS_DIR) for s in SOURCES}

In [ ]:
BACKBONE_DIR.mkdir(parents=True, exist_ok=True)
for a in ALPHAS:
    (BACKBONE_DIR / f"alpha{a}").mkdir(parents=True, exist_ok=True)

rows = []
for source in SOURCES:
    info_s = info[source]
    prev = {"disparity": None, "rca": None}

    for year in tqdm(info_s["years"], desc=source):
        W = load_npz(NETWORKS_DIR / source / f"adj_{year}.npz").tocsr()
        deg = np.diff(W.indptr).astype(float)
        s_orig = np.asarray(W.sum(axis=1)).ravel()
        act = s_orig > 0
        total_w = W.data.sum() / 2

        da = disparity_alpha(W)

        r = {"source": source, "year": int(year),
             "N_t": int(act.sum()), "E_t": W.nnz // 2}
        cur = {}

        bb = {}
        for m in ["disparity", "rca"]:
            d = extract_backbone(W, 3.0, m, scored=da if m == "disparity" else None)
            bb[m] = d
            sel = d[d.mst_only == 0]
            cur[m] = set(zip(d.i.to_numpy(), d.j.to_numpy()))

            if m == "disparity":
                d.to_parquet(BACKBONE_DIR / f"{source}_{year}_backbone.parquet")

            # does the backbone keep the original ranking of concepts?
            s_bb = np.zeros(W.shape[0])
            np.add.at(s_bb, d.i.to_numpy(), d.w.to_numpy())
            np.add.at(s_bb, d.j.to_numpy(), d.w.to_numpy())
            r[f"{m}_strength_corr"] = float(spearmanr(s_orig[act], s_bb[act]).correlation)

            # is the same structure picked up from one year to the next?
            r[f"{m}_year_jac"] = (len(cur[m] & prev[m]) / len(cur[m] | prev[m])
                                  if prev[m] else np.nan)

            r[f"{m}_wret"] = float(d.w.sum() / total_w)
            r[f"{m}_endpoint_deg"] = float((deg[sel.i] + deg[sel.j]).mean() / 2)
            r[f"{m}_E_bb"] = len(d)
            r[f"{m}_mean_degree"] = 2 * len(d) / len(np.unique(
                np.concatenate([d.i.to_numpy(), d.j.to_numpy()])))

        # the alpha the density rule implies, read off the backbone built above
        r["implied_alpha"] = float(bb["disparity"].query("mst_only == 0").alpha_min.max())

        for a in ALPHAS:
            sig = da[da.alpha_min < a]
            sig.to_parquet(BACKBONE_DIR / f"alpha{a}" / f"{source}_{year}_backbone.parquet")

            if len(sig):
                nd = np.unique(np.concatenate([sig.i.to_numpy(), sig.j.to_numpy()]))
                sub = csr_matrix((np.ones(len(sig)), (sig.i, sig.j)), shape=W.shape)
                nc, lab = connected_components(sub + sub.T, directed=False)
                sizes = np.bincount(lab)
                sizes = sizes[sizes > 1]

                r[f"a{a}_deg"] = 2 * len(sig) / len(nd)
                r[f"a{a}_cov"] = len(nd) / int(act.sum())
                r[f"a{a}_wret"] = float(sig.w.sum() / total_w)
                r[f"a{a}_giant"] = float(sizes.max() / sizes.sum()) if len(sizes) else 0.0
            else:
                r[f"a{a}_deg"] = r[f"a{a}_cov"] = 0.0
                r[f"a{a}_wret"] = r[f"a{a}_giant"] = 0.0

        r["jaccard"] = len(cur["disparity"] & cur["rca"]) / len(cur["disparity"] | cur["rca"])
        prev = cur
        rows.append(r)

bb_cmp = pd.DataFrame(rows)
bb_cmp.to_csv(BACKBONE_DIR / "filter_comparison.csv", index=False)
bb_cmp.head()

In [ ]:
bb_cmp.groupby("source").agg(
    jaccard=("jaccard", "median"),
    disp_strength=("disparity_strength_corr", "median"),
    rca_strength=("rca_strength_corr", "median"),
    disparity_year_jac=("disparity_year_jac", "median"),
    rca_year_jac=("rca_year_jac", "median"),
    disp_wret=("disparity_wret", "median"),
    rca_wret=("rca_wret", "median"),
    mean_degree=("disparity_mean_degree", "median"),
).round(3)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(8, 8), sharex=True)
C = {"disparity": "#FF1F5B", "rca": "#009ADE"}

for col, s in enumerate(SOURCES):
    d = bb_cmp[bb_cmp.source == s]

    # does it preserve the original hierarchy?
    ax = axes[0, col]
    for m in C:
        ax.plot(d.year, d[f"{m}_strength_corr"], color=C[m], lw=1.5, label=m)
    ax.set_ylim(0, 1)
    ax.set_title(s, fontsize=12)
    if col == 0:
        ax.set_ylabel("Strength rank correlation\nwith full network", fontsize=10)
        ax.legend(fontsize=9)

    # how much of the original weight survives?
    ax = axes[1, col]
    for m in C:
        ax.plot(d.year, d[f"{m}_wret"], color=C[m], lw=1.5)
    ax.set_ylim(0, 1)
    if col == 0:
        ax.set_ylabel("Weight retained", fontsize=10)

    # is the structure stable across years?
    ax = axes[2, col]
    for m in C:
        ax.plot(d.year, d[f"{m}_year_jac"], color=C[m], lw=1.5)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Year", fontsize=11)
    if col == 0:
        ax.set_ylabel("Jaccard with previous year", fontsize=10)

for ax in axes.flat:
    ax.set_xlim(1990, 2023)
    ax.tick_params(labelsize=9)

fig.suptitle("Disparity vs. RCA: what each backbone preserves", fontsize=13, y=0.99)
plt.tight_layout()
plt.savefig(PLOT_DIR / "backbone_disparity_vs_rca.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(8, 8), sharex=True)
AC = {0.01: "#FF1F5B", 0.05: "#00CD6C", 0.1: "#009ADE", 0.2: "#AF58BA"}
PANELS = [("deg", "Mean degree"),
          ("cov", "Node coverage"),
          ("wret", "Weight retained")]

for col, s in enumerate(SOURCES):
    d = bb_cmp[bb_cmp.source == s]
    for row, (key, ylab) in enumerate(PANELS):
        ax = axes[row, col]
        for a, c in AC.items():
            ax.plot(d.year, d[f"a{a}_{key}"], color=c, lw=1.4,
                    label=rf"$\alpha<{a}$")
        if key != "deg":
            ax.set_ylim(0, 1.02)
        if row == 0:
            ax.set_title(s, fontsize=12)
            if col == 0:
                ax.legend(fontsize=8, ncol=2)
        if col == 0:
            ax.set_ylabel(ylab, fontsize=10)
        if row == len(PANELS) - 1:
            ax.set_xlabel("Year", fontsize=11)
        ax.set_xlim(1990, 2023)
        ax.tick_params(labelsize=9)

fig.suptitle("Disparity filter: sensitivity to a fixed threshold", fontsize=13, y=0.995)
plt.tight_layout()
plt.savefig(PLOT_DIR / "backbone_alpha_sensitivity.png", dpi=300, bbox_inches="tight")
plt.show()

### Persistent path

In [ ]:
def persistent_edges(source, info, bb_dir, min_years=20):
    """
    Edges present in the backbone for at least min_years consecutive years.
    """
    years = list(info["years"])
    present = {}
    for t, year in enumerate(years):
        bb = pd.read_parquet(Path(bb_dir) / f"{source}_{year}_backbone.parquet")
        for i, j in zip(bb.i.to_numpy(), bb.j.to_numpy()):
            present.setdefault((i, j), np.zeros(len(years), bool))[t] = True

    out = []
    for (i, j), mask in present.items():
        run = best = 0
        for v in mask:
            run = run + 1 if v else 0
            best = max(best, run)
        if best >= min_years:
            out.append((i, j, best))
    return pd.DataFrame(out, columns=["i", "j", "run"])


def persistent_paths(pers_edges, length=8, seed=0):
    """
    One path per randomly drawn start node.
    """
    rng = np.random.default_rng(seed)
    adj = {}
    for i, j in zip(pers_edges.i, pers_edges.j):
        adj.setdefault(i, set()).add(j)
        adj.setdefault(j, set()).add(i)

    paths, seen = [], set()
    for s in rng.permutation(list(adj)):
        path = [int(s)]
        while len(path) < length:
            cand = list(adj[path[-1]] - set(path))
            if not cand:
                break
            path.append(int(rng.choice(cand)))
        if len(path) < length:
            continue
        key = tuple(sorted(path))
        if key in seen:
            continue
        seen.add(key)
        paths.append(path)
    return paths

In [ ]:
pe = {s: persistent_edges(s, info[s], BACKBONE_DIR, min_years=20) for s in SOURCES}
paths_seq = {s: persistent_paths(pe[s]) for s in SOURCES}

for s in SOURCES:
    nodes = sorted({n for p in paths_seq[s] for n in p})
    sub = pe[s][pe[s].i.isin(nodes) & pe[s].j.isin(nodes)]
    n = len(nodes)
    print(f"{s}: {len(pe[s]):,} persistent edges | {len(paths_seq[s]):,} paths | "
          f"{n:,} unique nodes | density {2*len(sub)/(n*(n-1)):.3f}")

with open(BACKBONE_DIR / "persistent_paths.pkl", "wb") as f:
    pickle.dump({"paths": paths_seq, "pers_edges": pe, "min_years": 20}, f)

In [ ]:
edge_sets = {}
for source in SOURCES:
    for a in ALPHAS:
        pe_a = persistent_edges(source, info[source], BACKBONE_DIR / f"alpha{a}",
                                min_years=20)
        edge_sets[(source, a)] = set(zip(pe_a.i, pe_a.j))
        print(f"{source} alpha<{a}: {len(pe_a):,} persistent edges")

In [ ]:
pers_alpha = pd.DataFrame([
    {"source": s, "alpha": a, "n_persistent": len(edge_sets[(s, a)])}
    for s in SOURCES for a in ALPHAS
])

# how much does the persistent set change with the threshold?
for s in SOURCES:
    base = edge_sets[(s, 0.05)]
    for a in ALPHAS:
        e = edge_sets[(s, a)]
        pers_alpha.loc[(pers_alpha.source == s) & (pers_alpha.alpha == a),
                       "jaccard_vs_005"] = (len(base & e) / len(base | e)
                                            if (base | e) else np.nan)

path_alpha, path_sets = [], {}
for source in SOURCES:
    for a in ALPHAS:
        pe_a = pd.DataFrame(sorted(edge_sets[(source, a)]), columns=["i", "j"])
        p = persistent_paths(pe_a, length=8, seed=0)
        path_sets[(source, a)] = {n for x in p for n in x}
        path_alpha.append({"source": source, "alpha": a,
                           "n_paths": len(p), "n_nodes": len(path_sets[(source, a)])})

path_alpha = pd.DataFrame(path_alpha)
for s in SOURCES:
    base = path_sets[(s, 0.05)]
    for a in ALPHAS:
        cur = path_sets[(s, a)]
        path_alpha.loc[(path_alpha.source == s) & (path_alpha.alpha == a),
                       "node_jaccard_vs_005"] = len(base & cur) / len(base | cur)

pers_alpha.merge(path_alpha, on=["source", "alpha"]).round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
C = {"news": "#FF1F5B", "paper": "#009ADE"}

ax = axes[0]
for s in SOURCES:
    d = pers_alpha[pers_alpha.source == s]
    ax.plot(d.alpha, d.n_persistent, color=C[s], lw=1.6, marker="o", ms=5, label=s)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_ylabel("Persistent edges", fontsize=10)
ax.legend(fontsize=9)
ax.set_title("$(a)$", loc="left")

ax = axes[1]
for s in SOURCES:
    d = path_alpha[path_alpha.source == s]
    ax.plot(d.alpha, d.n_paths, color=C[s], lw=1.6, marker="o", ms=5)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_ylabel("Persistent paths", fontsize=10)
ax.set_title("$(b)$", loc="left")

ax = axes[2]
for s in SOURCES:
    d = pers_alpha[pers_alpha.source == s]
    ax.plot(d.alpha, d.jaccard_vs_005, color=C[s], lw=1.6, marker="o", ms=5)
ax.set_xscale("log"); ax.set_ylim(0, 1.02)
ax.set_ylabel(r"Edge overlap with $\alpha<0.05$", fontsize=10)
ax.set_title("$(c)$", loc="left")

for ax in axes:
    ax.set_xlabel(r"$\alpha$ threshold", fontsize=11)
    ax.tick_params(labelsize=9)

plt.tight_layout()
plt.savefig(PLOT_DIR / "persistent_alpha_sensitivity.png", dpi=300, bbox_inches="tight")
plt.show()

### Perplexity

In [ ]:
def path_pseudo_perplexity(seq_paths, Z_t, active_idx, tau=1.0, normalize=True):
    """
    Mask one concept at a time and score it against every active concept,
    using the mean of the remaining concepts as context.
    """
    pos = {int(n): k for k, n in enumerate(active_idx)}
    Za = Z_t[active_idx]
    if normalize:
        Za = Za / (np.linalg.norm(Za, axis=1, keepdims=True) + 1e-12)

    rows = []
    for p in seq_paths:
        local = [pos[v] for v in p if v in pos]
        if len(local) < 2:
            continue

        logps = []
        for k in range(len(local)):
            ctx = np.delete(np.asarray(local), k)
            c = Za[ctx].mean(axis=0)
            s = Za @ c / tau
            logps.append(s[local[k]] - logsumexp(s))

        rows.append({"path": tuple(p),
                     "n_scored": len(local),
                     "ppl": float(np.exp(-np.mean(logps))),
                     "n_active": len(active_idx)})
    return pd.DataFrame(rows)


def run_pseudo_perplexity(source, info, seq_paths, emb_path=None, tau=1.0):
    """
    One row per (path, year). Embedding indices match the arena vocabulary.

    `emb_path` defaults to <embeddings>/<source>_E.npz, the file written by
    src/dysat/train.py. A bare .npy of shape [N, T, F] is also accepted; then the
    active set falls back to the yearly document frequency from notebook 01.
    """
    from scisoc.io import load_embeddings

    Z, emb_active = load_embeddings(source, emb_path)       # [N, T, F], [N, T] or None
    freq = np.asarray(info["concept_freq_year"])
    assert Z.shape[1] == len(info["years"]), (
        f"embedding has {Z.shape[1]} snapshots but node_info has {len(info['years'])} years")
    out = []
    for t, year in enumerate(tqdm(info["years"], desc=source)):
        active = np.where(emb_active[:, t] if emb_active is not None else freq[t] > 0)[0]
        d = path_pseudo_perplexity(seq_paths, Z[:, t, :], active, tau)
        d["source"], d["year"] = source, int(year)
        out.append(d)

    d = pd.concat(out, ignore_index=True)
    d["ppl_rel"] = d["ppl"] / d["n_active"]
    return d

In [ ]:
# Requires src/dysat/train.py to have written <embeddings>/<source>_E.npz.
# pp = run_pseudo_perplexity("news", info["news"], paths_seq["news"])
# pp.groupby("year")[["ppl", "ppl_rel", "n_active"]].median().round(4)
